In [1]:
import sympy as sp
from IPython.display import display, Math

def display_broken_equation(equation, terms_per_line=4):
    lhs = sp.latex(equation.lhs)

    rhs_terms = sp.Add.make_args(equation.rhs)

    lines = []

    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]
        expression = sp.Add(*group)

        latex_group = sp.latex(expression)

        if i == 0:
            lines.append(rf"{lhs} &= {latex_group}")
        else:
            lines.append(rf"&\quad {latex_group}")

    latex_output = (
        r"\begin{aligned}"
        + r" \\ ".join(lines)
        + r"\end{aligned}"
    )

    display(Math(latex_output))

In [2]:
def brokeq(equation, terms_per_line=4):
    lhs = sp.latex(equation.lhs)
    rhs_terms = list(sp.Add.make_args(equation.rhs))

    lines = []
    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]

        # Build the latex for this group manually so we control signs
        terms_latex = []
        for j, term in enumerate(group):
            coeff, rest = term.as_coeff_Mul()
            sign = "-" if coeff < 0 else "+"
            # strip sign from latex
            term_latex = sp.latex(abs(coeff) * rest) if coeff != 1 and coeff != -1 \
                         else sp.latex(rest)
            terms_latex.append((sign, term_latex))

        # First term of the whole RHS has no leading + / -
        first_sign, first_tex = terms_latex[0]
        if i == 0:
            head = first_tex if first_sign == "+" else f"- {first_tex}"
            line = rf"{lhs} &= {head}"
        else:
            head = first_tex if first_sign == "+" else f"- {first_tex}"
            prefix = "+ " if first_sign == "+" else ""
            # Actually for continuation we want the sign explicit:
            line = rf"&\quad {first_sign} {first_tex}"

        # Remaining terms in the group
        for sign, tex in terms_latex[1:]:
            line += rf" {sign} {tex}"

        lines.append(line)

    return r"\begin{aligned}" + r" \\ ".join(lines) + r"\end{aligned}"

In [24]:

import sympy as sp


def fdm_ordered_brokeq(equation, terms_per_line=4, symbols=None):
    """
    Sort and display a SymPy equation as multiline LaTeX.

    Time order:
        t+1, t, t-1, t-2, ...

    Space order within each time level:
        x, x+1, x-1, x+2, x-2, x+3, x-3, ...

    Parameters
    ----------
    equation : sympy.Equality
    terms_per_line : int
    symbols : dict, optional
        Dictionary mapping keys such as 'ut-1d+2'
        to SymPy symbols.
    """

    if terms_per_line < 1:
        raise ValueError("terms_per_line must be >= 1")

    lhs = sp.latex(equation.lhs)
    rhs_terms = list(sp.Add.make_args(equation.rhs))

    reverse = {
        value: key
        for key, value in (symbols or {}).items()
    }

    # -----------------------------------------------------
    # Interpret dictionary labels and LaTeX symbol names
    # -----------------------------------------------------

    def parse_label(label):
        label = str(label).replace(" ", "")

        # Dictionary labels: ut0, ut-1d+2, etc.
        if label.startswith("ut"):
            parts = label[2:].split("d")

            if len(parts) <= 2:
                try:
                    t = int(parts[0])
                    x = int(parts[1]) if len(parts) == 2 else 0
                    return t, x
                except ValueError:
                    pass

        # LaTeX names: u^{t}_{x}, u_{x}^{t}, etc.
        if label.startswith("u"):
            import re as regex

            time_match = regex.search(
                r"\^\{t([+-]\d+)?\}", label
            )

            space_match = regex.search(
                r"_\{x([+-]\d+)?\}", label
            )

            if time_match and space_match:
                t = int(time_match.group(1) or 0)
                x = int(space_match.group(1) or 0)
                return t, x

        return None

    # -----------------------------------------------------
    # Sort by time, then alternating spatial displacement
    # -----------------------------------------------------

    def sort_key(term):
        candidates = []

        for symbol in term.free_symbols:

            label = reverse.get(symbol, str(symbol))
            offsets = parse_label(label)

            if offsets is None:
                continue

            t, x = offsets

            # x, x+1, x-1, x+2, x-2, ...
            x_key = (abs(x), 0 if x >= 0 else 1)

            candidates.append((-t, *x_key))

        return min(
            candidates,
            default=(float("inf"), float("inf"), 2)
        )

    rhs_terms.sort(key=sort_key)

    # -----------------------------------------------------
    # Build LaTeX with explicit signs
    # -----------------------------------------------------

    def term_parts(term):
        coeff, rest = term.as_coeff_Mul()

        negative = coeff.is_negative is True
        sign = "-" if negative else "+"
        magnitude = -coeff if negative else coeff

        if rest == 1:
            tex = sp.latex(magnitude)
        elif magnitude == 1:
            tex = sp.latex(rest)
        else:
            tex = sp.latex(
                sp.Mul(magnitude, rest, evaluate=False)
            )

        return sign, tex

    lines = []

    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]
        parts = [term_parts(term) for term in group]

        first_sign, first_tex = parts[0]

        if i == 0:
            head = (
                first_tex if first_sign == "+"
                else f"- {first_tex}"
            )
            line = rf"{lhs} &= {head}"
        else:
            line = rf"&\quad {first_sign} {first_tex}"

        for sign, tex in parts[1:]:
            line += rf" {sign} {tex}"

        lines.append(line)

    return (
        r"\begin{aligned}"
        + r" \\ ".join(lines)
        + r"\end{aligned}"
    )


In [ ]:
def brokeq2(equation, terms_per_line=4):
    rhs_terms = sp.Add.make_args(equation)
    lines = []
    for i in range(0, len(rhs_terms), terms_per_line):
        group = rhs_terms[i:i + terms_per_line]
        expression = sp.Add(*group)
        latex_group = sp.latex(expression)
        lines.append(rf"&\quad {latex_group}")

    latex_output = (r"\begin{aligned}" + r" \\ ".join(lines) + r"\end{aligned}")

    display(Math(latex_output))